# EduShield AI — Dropout Risk Model (v2)

Trains and evaluates classifiers on the UCI Student Performance dataset (Cortez & Silva, 2008),
selects the final model by grouped cross-validation, and integrates SHAP for per-prediction
explanations.

**v2 — leakage fixes.** The v1 notebook (archived at `notebooks/archive/dropout_model_v1.ipynb`,
artifacts at `model/archive/xgboost-v1.0.0/`) over-stated performance for four reasons, each fixed
below:

1. **Label leakage.** v1 defined the label as `G3 < 10 OR failures >= 2` while also using
   `failures` as an input, so every student with 2+ past failures was at-risk by construction.
   v2 labels on the outcome only: `G3 < 10`. `failures` stays as an input because it is genuine
   history that is known before the outcome.
2. **The same students in train and test.** `student-mat.csv` and `student-por.csv` contain many
   of the same students (`data/student-merge.R` matches 382). v1 split rows at random, so one
   student's maths and Portuguese records could land on opposite sides. v2 groups rows by student
   and splits by group.
3. **Preprocessing fitted on all data.** v1 fitted the encoders and scaler before splitting. v2
   fits them on the training split only.
4. **Model chosen on the test set.** v1 picked the model with the best test-set F1. v2 picks it by
   grouped cross-validation on the training split and uses the test split exactly once, for the
   final figures.

Output artifacts feed `ml-service/app` and are the ground truth for `contracts/types.ts`'s
`Prediction` shape.

In [1]:
import pandas as pd
import numpy as np

mat = pd.read_csv('../data/student-mat.csv', sep=';')
por = pd.read_csv('../data/student-por.csv', sep=';')

mat['course'] = 'mat'
por['course'] = 'por'

df = pd.concat([mat, por], ignore_index=True)

# The two files describe many of the same students. These are the identifying attributes the
# dataset authors use to match them (data/student-merge.R). Rows sharing all of them are treated
# as one student, so a student can never appear on both sides of a train/test split.
MERGE_KEYS = ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
              'Mjob', 'Fjob', 'reason', 'nursery', 'internet']
df['student_group'] = df.groupby(MERGE_KEYS, sort=False).ngroup()

group_courses = df.groupby('student_group')['course'].nunique()
print(f"mat: {len(mat)} rows, por: {len(por)} rows, combined: {len(df)} rows")
print(f"distinct student groups: {df['student_group'].nunique()}")
print(f"groups present in both courses: {(group_courses > 1).sum()}")

mat: 395 rows, por: 649 rows, combined: 1044 rows
distinct student groups: 662
groups present in both courses: 366


## Target label

```
dropout_risk = 1  if  G3 < 10   (fails the 0-20 pass mark)
             = 0  otherwise
```

- `G3` defines the label, so it is excluded from the inputs.
- `failures` (past class failures) is an input. It describes the student's history before the
  current year, so using it does not leak the outcome. In v1 it was also part of the label, which
  made the label partly a copy of an input; that is removed.
- `G1` and `G2` (first- and second-period grades) are inputs. They are known before the final
  grade, but `G2` arrives late in the year, so the sensitivity analysis further down reports how
  much of the performance depends on them.
- The dataset comes from two Portuguese secondary schools, and the label is "fails the final
  grade", which is a proxy for dropout risk rather than recorded dropout.

In [2]:
df['dropout_risk'] = (df['G3'] < 10).astype(int)
print(df['dropout_risk'].value_counts(normalize=True).round(3))

# Contract's StudentFeatures set (contracts/types.ts) — G3 deliberately excluded.
FEATURE_COLUMNS = [
    'age', 'sex', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities',
    'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc',
    'health', 'absences', 'Medu', 'Fedu', 'G1', 'G2',
]
TARGET_COLUMN = 'dropout_risk'
assert 'G3' not in FEATURE_COLUMNS

# Past failures are strongly associated with failing, but no longer determine the label.
print(df.groupby('failures')[TARGET_COLUMN].agg(at_risk_rate='mean', students='size').round(2))

X_raw = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()
groups = df['student_group']

dropout_risk
0    0.78
1    0.22
Name: proportion, dtype: float64
          at_risk_rate  students
failures                        
0                 0.15       861
1                 0.48       120
2                 0.67        33
3                 0.70        30


## Train/test split by student, preprocessing fitted on training data only

One fold of a 5-fold `StratifiedGroupKFold` is held out as the test split (about 20% of rows).
Stratification keeps the at-risk proportion similar on both sides, and grouping keeps every
student's records on one side only.

`sex` and the `yes`/`no` columns are label-encoded to 0/1 and the numeric columns are scaled with
`StandardScaler`, both fitted on the training split and then applied unchanged to the test split
(and, in production, by `ml-service/app/predictor.py`).

In [3]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler

outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(outer.split(X_raw, y, groups))

X_train_raw, X_test_raw = X_raw.iloc[train_idx], X_raw.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

assert set(groups_train).isdisjoint(set(groups.iloc[test_idx])), 'a student appears in both splits'

BINARY_YES_NO = ['schoolsup', 'famsup', 'paid', 'activities', 'higher', 'internet', 'romantic']
CATEGORICAL = ['sex'] + BINARY_YES_NO
NUMERIC = [c for c in FEATURE_COLUMNS if c not in CATEGORICAL]

encoders = {col: LabelEncoder().fit(X_train_raw[col]) for col in CATEGORICAL}
scaler = StandardScaler().fit(X_train_raw[NUMERIC].astype(float))


def preprocess(frame):
    out = frame.astype({col: float for col in NUMERIC})
    for col in CATEGORICAL:
        out[col] = encoders[col].transform(out[col])
    out[NUMERIC] = scaler.transform(out[NUMERIC])
    return out


X_train = preprocess(X_train_raw)
X_test = preprocess(X_test_raw)

print(f"train: {X_train.shape}, test: {X_test.shape}")
print(f"at-risk rate: train {y_train.mean():.3f}, test {y_test.mean():.3f}")

train: (835, 22), test: (209, 22)
at-risk rate: train 0.220, test 0.220


## Feature selection via correlation analysis

Per the proposal's Step 2 ("perform feature selection using correlation analysis"), computed on
the training split only. Two checks:

1. **Multicollinearity** — pairs of input features correlated above `|r| > 0.7`.
2. **Target relevance** — features with near-zero correlation to `dropout_risk`.

In [4]:
corr_df = X_train.copy()
corr_df['dropout_risk'] = y_train.values
corr_matrix = corr_df.corr()

feature_corr = corr_matrix.drop(columns='dropout_risk').drop(index='dropout_risk')
high_corr_pairs = []
for i, a in enumerate(feature_corr.columns):
    for b in feature_corr.columns[i + 1:]:
        r = feature_corr.loc[a, b]
        if abs(r) > 0.7:
            high_corr_pairs.append((a, b, round(float(r), 3)))

print("Multicollinear feature pairs (|r| > 0.7):", high_corr_pairs or "none found")

target_corr = corr_matrix['dropout_risk'].drop('dropout_risk').sort_values(key=abs)
print("\nWeakest target correlations:")
print(target_corr.head(6).round(4))
print("\nStrongest target correlations:")
print(target_corr.tail(6).round(4))

Multicollinear feature pairs (|r| > 0.7): [('G1', 'G2', 0.864)]

Weakest target correlations:
famsup        0.0025
activities   -0.0131
schoolsup     0.0151
health        0.0439
famrel       -0.0516
sex           0.0563
Name: dropout_risk, dtype: float64

Strongest target correlations:
age         0.1348
absences    0.1358
higher     -0.2344
failures    0.3768
G1         -0.6144
G2         -0.6711
Name: dropout_risk, dtype: float64


**Decision: no features removed.**

- **G1/G2** are the only strongly correlated pair. Both are kept: tree ensembles split on one
  feature at a time and tolerate correlated inputs, and the two grades carry distinct timing
  information. Their combined contribution is measured in the sensitivity analysis below.
- **Weakly correlated features** are kept because Pearson correlation only detects linear
  relationships, and tree ensembles can use non-linear interactions.

## Model comparison by grouped cross-validation

Four models are compared with 5-fold `StratifiedGroupKFold` cross-validation on the training split
only (students never shared between folds). Logistic Regression and Decision Tree are baselines.
Only Random Forest and XGBoost are eligible for deployment, because `ml-service` explains
predictions with `shap.TreeExplainer`. The deployed model is the eligible one with the higher mean
cross-validated F1 — the test split plays no part in the choice.

In [5]:
import warnings

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

cv_splits = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=7).split(X_train, y_train, groups_train))
SCORING = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
DEPLOYABLE = ['random_forest', 'xgboost']  # ml-service explains with shap.TreeExplainer


def make_candidates():
    return {
        'logistic_regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
        'decision_tree': DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42),
        'random_forest': RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, class_weight='balanced'),
        'xgboost': XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42, eval_metric='logloss'),
    }


def cross_validated(model, features):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        scores = cross_validate(model, X_train[features], y_train, cv=cv_splits, scoring=SCORING)
    return {m: scores[f'test_{m}'].mean() for m in SCORING} | {f'{m}_std': scores[f'test_{m}'].std() for m in SCORING}


cv_table = pd.DataFrame({name: cross_validated(model, FEATURE_COLUMNS) for name, model in make_candidates().items()}).T
print("Mean over 5 grouped folds (training split only):")
print(cv_table[SCORING].round(4))
print("\nStandard deviation across folds:")
print(cv_table[[f'{m}_std' for m in SCORING]].round(4))

Mean over 5 grouped folds (training split only):
                     accuracy  precision  recall      f1  roc_auc
logistic_regression    0.9018     0.7264  0.8967  0.8013   0.9694
decision_tree          0.8766     0.6807  0.8535  0.7545   0.9137
random_forest          0.9113     0.7549  0.8968  0.8178   0.9676
xgboost                0.9137     0.8347  0.7667  0.7965   0.9621

Standard deviation across folds:
                     accuracy_std  precision_std  recall_std  f1_std  \
logistic_regression        0.0121         0.0407      0.0316  0.0198   
decision_tree              0.0234         0.0606      0.0399  0.0320   
random_forest              0.0201         0.0598      0.0395  0.0356   
xgboost                    0.0052         0.0490      0.0458  0.0079   

                     roc_auc_std  
logistic_regression       0.0035  
decision_tree             0.0128  
random_forest             0.0050  
xgboost                   0.0051  


In [6]:
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)

best_name = cv_table.loc[DEPLOYABLE, 'f1'].idxmax()
best_model = make_candidates()[best_name]
best_model.fit(X_train, y_train)

# The held-out test split is used exactly once, here, for the selected model's final figures.
test_pred = best_model.predict(X_test)
test_proba = best_model.predict_proba(X_test)[:, 1]
test_metrics = {
    'accuracy': float(accuracy_score(y_test, test_pred)),
    'precision': float(precision_score(y_test, test_pred)),
    'recall': float(recall_score(y_test, test_pred)),
    'f1': float(f1_score(y_test, test_pred)),
    'auc_roc': float(roc_auc_score(y_test, test_proba)),
    'confusion_matrix': confusion_matrix(y_test, test_pred).tolist(),
}

print(f"Selected by cross-validated F1: {best_name}")
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in test_metrics.items()})
print(classification_report(y_test, test_pred))
print("Proposal target of >80% accuracy met:", test_metrics['accuracy'] > 0.80)

Selected by cross-validated F1: random_forest
{'accuracy': 0.9091, 'precision': 0.7368, 'recall': 0.913, 'f1': 0.8155, 'auc_roc': 0.9711, 'confusion_matrix': [[148, 15], [4, 42]]}
              precision    recall  f1-score   support

           0       0.97      0.91      0.94       163
           1       0.74      0.91      0.82        46

    accuracy                           0.91       209
   macro avg       0.86      0.91      0.88       209
weighted avg       0.92      0.91      0.91       209

Proposal target of >80% accuracy met: True


## Sensitivity: how much of the performance comes from the term grades?

`G2` arrives late in the school year, so a model that relies on it gives less time to intervene.
The selected model type is re-evaluated with the same grouped folds after removing the grade
features, to show how early a useful warning is possible.

In [7]:
FEATURE_SETS = {
    'all features (G1 + G2)': FEATURE_COLUMNS,
    'without G2 (first-period grade only)': [c for c in FEATURE_COLUMNS if c != 'G2'],
    'without G1 and G2 (no grades)': [c for c in FEATURE_COLUMNS if c not in ('G1', 'G2')],
}
sensitivity = pd.DataFrame({
    label: cross_validated(make_candidates()[best_name], cols) for label, cols in FEATURE_SETS.items()
}).T
print(f"{best_name}, mean over the same 5 grouped folds:")
print(sensitivity[SCORING].round(4))

random_forest, mean over the same 5 grouped folds:
                                      accuracy  precision  recall      f1  \
all features (G1 + G2)                  0.9113     0.7549  0.8968  0.8178   
without G2 (first-period grade only)    0.8275     0.5865  0.8044  0.6699   
without G1 and G2 (no grades)           0.7749     0.5010  0.4404  0.4593   

                                      roc_auc  
all features (G1 + G2)                 0.9676  
without G2 (first-period grade only)   0.9146  
without G1 and G2 (no grades)          0.7299  


## SHAP explainability

`TreeExplainer` works for both Random Forest and XGBoost. Raw SHAP values are normalized into the
`contracts/types.ts` `ShapFactor.contributionPct` convention:

```
contributionPct(feature) = |shap_value(feature)| / sum(|shap_value(f)| for f in all features) * 100
direction = 'increases' if shap_value(feature) > 0 else 'decreases'
```

In [8]:
import shap

explainer = shap.TreeExplainer(best_model)

FEATURE_LABELS = {
    'absences': 'Attendance', 'G1': 'First-term grade', 'G2': 'Second-term grade',
    'failures': 'Past failures', 'studytime': 'Study time', 'goout': 'Social outings',
    'Dalc': 'Weekday alcohol use', 'Walc': 'Weekend alcohol use', 'health': 'Health status',
    'famsup': 'Family support', 'schoolsup': 'School support', 'internet': 'Internet access',
    'higher': 'Aspiration for higher education', 'romantic': 'Romantic relationship',
    'famrel': 'Family relationship quality', 'freetime': 'Free time', 'age': 'Age',
    'sex': 'Gender', 'Medu': "Mother's education", 'Fedu': "Father's education",
    'activities': 'Extra-curricular activities', 'paid': 'Paid extra classes',
}

EXPLANATION_TEMPLATES = {
    'absences': ('High absence count is strongly associated with dropout risk.',
                 'Consistent attendance is acting as a protective factor.'),
    'G1': ('A weak first-term grade is an early warning sign of risk.',
           'A strong first-term grade is acting as a protective factor.'),
    'G2': ('A weak second-term grade is an early warning sign of risk.',
           'A strong second-term grade is acting as a protective factor.'),
    'failures': ('Previous module failures are associated with a higher probability of withdrawal.',
                 'No prior failures is acting as a protective factor.'),
    'studytime': ('Low weekly study time increases risk.', 'Consistent study time is protective.'),
    'goout': ('Frequent social outings correlate with reduced academic focus.', 'Balanced social activity is not a risk driver here.'),
}


def positive_class_shap(raw):
    # TreeExplainer's output shape depends on model and shap version: a per-class list,
    # an (n, features, classes) array, or an (n, features) array for the positive class.
    if isinstance(raw, list):
        raw = raw[1]
    raw = np.asarray(raw)
    if raw.ndim == 3:
        raw = raw[..., 1]
    return raw[0]


def normalize_shap(raw_values, feature_row, feature_columns):
    # Raw per-feature SHAP values -> contracts/types.ts ShapFactor[] convention.
    total_abs = np.sum(np.abs(raw_values)) or 1e-9
    factors = []
    for i, col in enumerate(feature_columns):
        contribution_pct = float(abs(raw_values[i]) / total_abs * 100)
        direction = 'increases' if raw_values[i] > 0 else 'decreases'
        risk_text, protective_text = EXPLANATION_TEMPLATES.get(
            col, ('This factor is increasing the predicted risk.', 'This factor is reducing the predicted risk.')
        )
        factors.append({
            'feature': FEATURE_LABELS.get(col, col),
            'rawFeature': col.lower(),  # contracts/types.ts ShapFactor.rawFeature is a StudentFeatures key (lowercase)
            'observedValue': feature_row[col],
            'contributionPct': round(contribution_pct, 2),
            'direction': direction,
            'explanation': risk_text if direction == 'increases' else protective_text,
        })
    factors.sort(key=lambda f: f['contributionPct'], reverse=True)
    return factors


# Demo: explain one at-risk student from the held-out test split
sample_idx = y_test[y_test == 1].index[0]
sample_row = X_test.loc[[sample_idx]]
raw_shap = positive_class_shap(explainer.shap_values(sample_row))

sample_factors = normalize_shap(raw_shap, df.loc[sample_idx], FEATURE_COLUMNS)
sample_factors[:5]

[{'feature': 'Second-term grade',
  'rawFeature': 'g2',
  'observedValue': np.int64(6),
  'contributionPct': 50.5,
  'direction': 'increases',
  'explanation': 'A weak second-term grade is an early warning sign of risk.'},
 {'feature': 'First-term grade',
  'rawFeature': 'g1',
  'observedValue': np.int64(5),
  'contributionPct': 28.65,
  'direction': 'increases',
  'explanation': 'A weak first-term grade is an early warning sign of risk.'},
 {'feature': 'Past failures',
  'rawFeature': 'failures',
  'observedValue': np.int64(0),
  'contributionPct': 3.77,
  'direction': 'decreases',
  'explanation': 'No prior failures is acting as a protective factor.'},
 {'feature': 'Social outings',
  'rawFeature': 'goout',
  'observedValue': np.int64(4),
  'contributionPct': 2.47,
  'direction': 'increases',
  'explanation': 'Frequent social outings correlate with reduced academic focus.'},
 {'feature': 'Family support',
  'rawFeature': 'famsup',
  'observedValue': 'no',
  'contributionPct': 1.69,
 

In [9]:
RISK_TIER_THRESHOLDS = {'HIGH': 71, 'MEDIUM': 31, 'LOW': 0}


def risk_score_and_tier(proba_at_risk):
    # 0-1 model probability -> (0-100 riskScore, RiskTier), using RISK_TIER_THRESHOLDS.
    score = round(float(proba_at_risk) * 100, 1)
    tier = 'HIGH' if score >= RISK_TIER_THRESHOLDS['HIGH'] else 'MEDIUM' if score >= RISK_TIER_THRESHOLDS['MEDIUM'] else 'LOW'
    return score, tier


sample_proba = best_model.predict_proba(sample_row)[0, 1]
print(risk_score_and_tier(sample_proba))

(88.0, 'HIGH')


## Intervention rule table

Generic risk-factor → recommendation mapping (`contracts/types.ts` `InterventionRule[]`), per the
proposal's Step 5. `triggerFeature` uses the lowercase `StudentFeatures` keys, the same casing as
`ShapFactor.rawFeature`, so the backend's intervention engine can match them directly.

In [10]:
import json

INTERVENTION_RULES = [
    # Tier-level defaults (always fire for that tier) — matches proposal Step 5 verbatim.
    {'tier': 'HIGH', 'triggerFeature': 'default', 'action': 'Schedule a counseling session within 48 hours'},
    {'tier': 'HIGH', 'triggerFeature': 'default', 'action': 'Assign an academic mentor'},
    {'tier': 'MEDIUM', 'triggerFeature': 'default', 'action': 'Set up weekly attendance monitoring'},
    {'tier': 'MEDIUM', 'triggerFeature': 'default', 'action': 'Suggest joining a study group'},
    {'tier': 'LOW', 'triggerFeature': 'default', 'action': 'Provide positive reinforcement'},
    {'tier': 'LOW', 'triggerFeature': 'default', 'action': 'Continue regular check-ins'},
    # Feature-triggered additions — only added when that SHAP factor is a top risk-increasing driver.
    {'tier': 'HIGH', 'triggerFeature': 'absences', 'action': 'Arrange an urgent attendance review meeting'},
    {'tier': 'HIGH', 'triggerFeature': 'failures', 'action': 'Refer to academic recovery / repeat-module support'},
    {'tier': 'HIGH', 'triggerFeature': 'dalc', 'action': 'Refer to student wellbeing and counseling services'},
    {'tier': 'HIGH', 'triggerFeature': 'walc', 'action': 'Refer to student wellbeing and counseling services'},
    {'tier': 'MEDIUM', 'triggerFeature': 'g1', 'action': 'Recommend tutoring or supplementary academic support'},
    {'tier': 'MEDIUM', 'triggerFeature': 'g2', 'action': 'Recommend tutoring or supplementary academic support'},
    {'tier': 'MEDIUM', 'triggerFeature': 'famsup', 'action': 'Connect with student support services for family support options'},
    {'tier': 'MEDIUM', 'triggerFeature': 'studytime', 'action': 'Refer to a study-skills workshop'},
]

with open('../model/intervention_rules.json', 'w') as f:
    json.dump(INTERVENTION_RULES, f, indent=2)

len(INTERVENTION_RULES)

14

## Serialize artifacts

Everything `ml-service/app` needs to load at boot and serve `POST /predict` without re-running this
notebook. `metrics` are the held-out test-split figures; `cvMetrics` are the cross-validation means
used for model selection.

In [11]:
import joblib
import os

os.makedirs('../model', exist_ok=True)
MODEL_VERSION = f"{best_name}-v2.0.0"

joblib.dump(best_model, '../model/model.joblib')
joblib.dump(scaler, '../model/scaler.joblib')
joblib.dump(encoders, '../model/encoders.joblib')

with open('../model/metadata.json', 'w') as f:
    json.dump({
        'modelVersion': MODEL_VERSION,
        'modelType': best_name,
        'labelDefinition': 'dropout_risk = G3 < 10',
        'evaluation': 'Held-out test split (~20%), split by student with StratifiedGroupKFold; '
                      'preprocessing fitted on the training split; model selected by 5-fold grouped CV F1 on the training split.',
        'featureColumns': FEATURE_COLUMNS,
        'categoricalColumns': CATEGORICAL,
        'numericColumns': NUMERIC,
        'metrics': test_metrics,
        'cvMetrics': {name: {m: round(float(cv_table.loc[name, m]), 4) for m in SCORING} for name in cv_table.index},
        'gradeSensitivityCv': {label: {m: round(float(sensitivity.loc[label, m]), 4) for m in SCORING} for label in sensitivity.index},
        'riskTierThresholds': RISK_TIER_THRESHOLDS,
        'featureLabels': FEATURE_LABELS,
    }, f, indent=2)

print(f"Saved model.joblib, scaler.joblib, encoders.joblib, metadata.json, intervention_rules.json under ml-service/model/ (version {MODEL_VERSION})")

Saved model.joblib, scaler.joblib, encoders.joblib, metadata.json, intervention_rules.json under ml-service/model/ (version random_forest-v2.0.0)


## Summary

- Dataset: UCI Student Performance (student-mat.csv + student-por.csv), 1,044 rows, grouped by
  student so that the same student never appears in both training and test data.
- Label: `dropout_risk = G3 < 10`. `failures`, `G1` and `G2` are inputs; `G3` is not.
- Preprocessing (encoders, scaler) fitted on the training split only.
- Models: Logistic Regression and Decision Tree baselines, Random Forest and XGBoost candidates,
  compared by 5-fold grouped cross-validation; the deployed model is the eligible candidate with
  the best mean CV F1. The held-out test split is used once, for the final figures above.
- Explainability: SHAP `TreeExplainer`, normalized to `ShapFactor.contributionPct` + `direction`.
- Artifacts written to `ml-service/model/`; the v1 artifacts are kept in `model/archive/xgboost-v1.0.0/`.